# Fruit data: exploration, correction and first models

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/gromicho/teaching/blob/main/courses/abw/notebooks/lecture-3/fruit-data-exploration.ipynb) [![Open in Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/gromicho/teaching/main?urlpath=tree/courses/abw/notebooks/lecture-3/fruit-data-exploration.ipynb)

ABW lecture companion, originally developed by the ABW teaching team. This edition preserves the original sequence of data inspection, models and interpretation. Predict each result before running the cell.


## Prepare the libraries and original teaching data
Install missing libraries, then load the verified raw fruit table. Corrections happen in the lesson below, after inspecting the observations.


In [ ]:
# Load the shared teaching utilities from this checkout or a verified download.
from pathlib import Path
import hashlib
import sys
from urllib.request import urlopen

support_path = next((folder / 'support' for folder in [Path.cwd(), *Path.cwd().parents]
                     if (folder / 'support' / 'teaching_utils.py').is_file()), None)
if support_path is None:
    support_path = Path.cwd() / '.teaching-support'
    support_path.mkdir(exist_ok=True)
    helper = support_path / 'teaching_utils.py'
    expected = 'fbfa41e12709a01548e213abba976dad1e21d0abcfd798a2dfd669706cc152bb'
    if not helper.exists() or hashlib.sha256(helper.read_bytes()).hexdigest() != expected:
        url = 'https://raw.githubusercontent.com/gromicho/teaching/f3ad11b77cae7dd05315d314c7e72ee8516aaa3d/support/teaching_utils.py'
        content = urlopen(url, timeout=45).read()
        if hashlib.sha256(content).hexdigest() != expected:
            raise ValueError('Teaching helper version changed; reopen the current course notebook.')
        helper.write_bytes(content)
sys.path.insert(0, str(support_path))
from teaching_utils import ensure_packages

required_packages = {'matplotlib': 'matplotlib', 'pandas': 'pandas', 'numpy': 'numpy', 'seaborn': 'seaborn', 'scipy': 'scipy', 'sklearn': 'scikit-learn'}
ensure_packages(required_packages)


In [ ]:
from pathlib import Path
from urllib.request import urlopen
import hashlib

# Prefer the checked-in file locally; Colab downloads the same frozen edition.
data_path = next((p for p in [Path("data/fruits.csv"), Path("fruits.csv")]
                 if p.is_file()), Path("fruits.csv"))
if not data_path.is_file():
    url = "https://raw.githubusercontent.com/gromicho/teaching/main/data/fruits.csv"
    with urlopen(url, timeout=45) as response:
        payload = response.read()
    if hashlib.sha256(payload).hexdigest() != "84235de12e6a8eb094423436b3cdd045c8b55288d773053c821d88db06443c22":
        raise ValueError("Dataset checksum mismatch; do not use an unverified copy.")
    data_path.write_bytes(payload)
assert hashlib.sha256(data_path.read_bytes()).hexdigest() == "84235de12e6a8eb094423436b3cdd045c8b55288d773053c821d88db06443c22", "Unexpected local data version"


In [ ]:
import pandas as pd, seaborn as sns

# Get the data

In [ ]:
fruits = pd.read_csv(data_path, delimiter=';', decimal=',')
fruits

# Visualize the data

In [ ]:
sns.lmplot(x='Length', y='Width', data=fruits, hue='Name', fit_reg=False)

# 'Fix' the outliers
These two corrections represent known entry mistakes in this teaching dataset: swapped dimensions and a factor-of-ten unit error. An unusual observation alone does not establish a mistake. The extrema identify row labels (`idxmin`/`idxmax`), which remain valid with a non-default index. Apply these corrections once to the raw table.


In [ ]:
idx_min_length = fruits.Length.idxmin()
idx_max_width = fruits.Width.idxmax()

In [ ]:
fruits.at[ idx_max_width, 'Length' ],fruits.at[ idx_max_width, 'Width' ] = fruits.at[ idx_max_width, 'Width' ],fruits.at[ idx_max_width, 'Length' ]

In [ ]:
fruits.at[ idx_min_length, 'Length' ] = fruits.at[ idx_min_length, 'Length' ] * 10
fruits.at[ idx_min_length, 'Width' ] = fruits.at[ idx_min_length, 'Width' ] * 10

In [ ]:
sns.lmplot(x='Length', y='Width', data=fruits, hue='Name', fit_reg=False)

# Classification trees

In [ ]:
from sklearn import tree
import matplotlib.pyplot as plt
features = ['Length','Width']

In [ ]:
clf = tree.DecisionTreeClassifier(criterion='entropy', max_depth=2, random_state=0).fit(fruits[features], fruits.Name )
plt.figure(figsize=(12,12))  # set plot size (denoted in inches)
_ = tree.plot_tree(clf, filled=True, fontsize=10, feature_names=features, class_names=sorted( fruits.Name.unique() ) )

In [ ]:
fruits.Name.value_counts()

In [ ]:
fruits[ clf.predict( fruits[features] ) != fruits.Name ]

# Added to explain better the concept of entropy

In [ ]:
from math import log2

def Entropy( p ):
  return sum( [ -p*log2(p) if p > 0 else 0 for p in p ])

def EntropyNormalized( p ):
  return Entropy( p ) / log2(len(p))

from scipy.stats import entropy

In [ ]:
p = [ .5, .5 ]
entropy(p,base=2),EntropyNormalized(p),Entropy(p)

In [ ]:
counts = fruits.Name.value_counts().to_dict()
count = counts.values()
sum(count)

In [ ]:
p = [ c/sum(count) for c in count]

In [ ]:
entropy(p,base=2),EntropyNormalized(p),Entropy(p)

In [ ]:
node = [0,20,25,2]
p = [ c/sum(node) for c in node]

In [ ]:
entropy(p,base=2),EntropyNormalized(p),Entropy(p)

In [ ]:
4/40*Entropy( [4/4, 0/4] )+36/40*Entropy( [16/36, 20/36] )

In [ ]:
Entropy( [16/36, 20/36] )

# Clustering

In [ ]:
from sklearn.cluster import KMeans

In [ ]:
kmeans = KMeans(n_clusters=2, n_init=10, random_state=0).fit(fruits[features])

In [ ]:
fruits['cluster'] = kmeans.labels_

In [ ]:
sns.lmplot(x='Length', y='Width', data=fruits, hue='cluster', fit_reg=False)

In [ ]:
sns.lmplot(x='Length', y='Width', data=fruits, hue='Name', fit_reg=False)

# Linear regression

In [ ]:
from sklearn.linear_model import LinearRegression

In [ ]:
lin_reg = LinearRegression()  # Create linear regression object
x_train = fruits[fruits.cluster==0].Length.values.reshape(-1,1)
y_train = fruits[fruits.cluster==0].Width.values.reshape(-1,1)
lin_reg.fit( x_train, y_train )  # Train the model using the training sets

In [ ]:
model_line = lin_reg.predict(x_train)
plt.scatter(x_train, y_train, color='black')
plt.plot(x_train, model_line, color='blue', linewidth=3)
plt.xticks(())
plt.yticks(())
plt.show()

In [ ]:
# Intercept: the value for y when x=0 for the predicted line, \beta_0 in the formulas
lin_reg.intercept_

In [ ]:
# Coefficient: the slope of the predicted line, \beta_1 in the formulas
lin_reg.coef_

In [ ]:
import numpy as np
assert np.isclose(Entropy([0.5, 0.5]), 1)
assert np.isclose(Entropy([0, 1]), 0)
assert np.isclose(Entropy(p), entropy(p, base=2))
